In [1]:
import pandas as pd 
import numpy as np
import joblib
from sklearn.compose import ColumnTransformer 
from sklearn.preprocessing import OneHotEncoder,StandardScaler 
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score 



In [2]:
#  load dataset
dataset = pd.read_csv(r"D:\python\Machine Learning and Data Science Capstone Project\2.Data preprocessing\cleaned_saas_sales.csv")


In [3]:
dataset

,Country,City,Region,Subregion,Customer,Industry,Segment,Product,Sales,Quantity,Discount,Profit,Year,Month,Day
0,Ireland,Dublin,EMEA,UKIR,Chevron,Energy,SMB,Marketing Suite,261.9600,2.0,0.00,41.9136,2022,11,9
1,Ireland,Dublin,EMEA,UKIR,Chevron,Energy,SMB,FinanceHub,731.9400,3.0,0.00,219.5820,2022,11,9
2,United States,New York City,AMER,NAMER,Phillips 66,Energy,Strategic,FinanceHub,14.6200,2.0,0.00,6.8714,2022,6,13
3,Germany,Stuttgart,EMEA,EU-WEST,Royal Dutch Shell,Energy,SMB,ContactMatcher,957.5775,5.0,0.45,-383.0310,2021,10,11
4,Germany,Stuttgart,EMEA,EU-WEST,Royal Dutch Shell,Energy,SMB,Marketing Suite - Gold,22.3680,2.0,0.20,2.5164,2021,10,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9989,Germany,Hamburg,EMEA,EU-WEST,Johnson & Johnson,Healthcare,SMB,SaaS Connector Pack,25.2480,3.0,0.20,4.1028,2020,1,22
9990,United States,Milwaukee,AMER,NAMER,American Express,Finance,SMB,SaaS Connector Pack,91.9600,2.0,0.00,15.6332,2023,2,27
9991,United States,Milwaukee,AMER,NAMER,American Express,Finance,SMB,Site Analytics,258.5760,2.0,0.20,19.3932,2023,2,27
9992,United States,Milwaukee,AMER,NAMER,American Express,Finance,SMB,Support,29.6000,4.0,0.00,13.3200,2023,2,27


In [4]:
X=dataset.drop("Profit",axis=1)
Y=dataset["Profit"]

In [5]:
X.shape

(9994, 14)

In [6]:
# categorical columns 
cat_cols = [
    'Country',
    'City',
    'Region',
    'Subregion',
    'Customer',
    'Industry',
    'Segment',
    'Product'
]

In [7]:
# one hot encoding 
preprocessor=ColumnTransformer(transformers=[("cat",OneHotEncoder(handle_unknown='ignore',
                sparse_output=False),cat_cols)],remainder="passthrough",force_int_remainder_cols=False)
X_encoded=preprocessor.fit_transform(X)
feature_names=preprocessor.get_feature_names_out()
print("Encoded Shape:",X_encoded.shape)

Encoded Shape: (9994, 457)


In [8]:
# Scalling
scaler=StandardScaler()
X_scaled=scaler.fit_transform(X_encoded)

In [9]:
# RFE using Linear Regression
rfe=RFE(estimator=LinearRegression(),n_features_to_select=20,step=10)
X_rfe=rfe.fit_transform(X_scaled,Y)
selected_features=feature_names[rfe.support_]
print("\nSelected Features:")
for feature in selected_features:
    print(feature)



Selected Features:
cat__City_Berlin
cat__City_Gothenburg
cat__City_Mumbai
cat__City_Osaka
cat__City_Tijuana
cat__Region_AMER
cat__Customer_Allstate
cat__Customer_Coca-Cola
cat__Customer_Costco Wholesale
cat__Customer_Nissan Motor
cat__Customer_Valero Energy
cat__Product_Alchemy
cat__Product_Big Ol Database
cat__Product_ContactMatcher
cat__Product_FinanceHub
cat__Product_Marketing Suite
cat__Product_Marketing Suite - Gold
remainder__Sales
remainder__Quantity
remainder__Discount


In [10]:
# final pridiction model
final_model=DecisionTreeRegressor(random_state=0)
final_model.fit(X_rfe,Y)

DecisionTreeRegressor(random_state=0)

In [11]:
# training score
train_pred=final_model.predict(X_rfe)
train_r2=r2_score(Y,train_pred)
print("\nTraining R² Score :", train_r2)


Training R² Score : 0.9998297636077648


In [12]:
# save everything
joblib.dump(preprocessor, "preprocessor.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(rfe, "rfe_selector.pkl")
joblib.dump(final_model, "profit_prediction_model.pkl")
print("\nModel Saved Successfully")



Model Saved Successfully
